# Validação do wrapper SAM 2.1 Hiera Small

Este notebook usa somente uma imagem RGB sintética para validar bounding box e pontos positivos/negativos. Não há download automático, imagem clínica ou interpretação do score como confiança clínica.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents) if (path / "backend").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Execute o notebook dentro da árvore do CarcinoIndex.")
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from ai.segmentation import SAM2Segmenter

# Ajuste esta variável caso o clone oficial esteja em outro diretório.
CHECKPOINT_PATH = PROJECT_ROOT.parent / "sam2" / "checkpoints" / "sam2.1_hiera_small.pt"
MODEL_CONFIG = "configs/sam2.1/sam2.1_hiera_s.yaml"
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError("Configure CHECKPOINT_PATH para o checkpoint oficial Small já baixado.")

In [ ]:
size = 512
yy, xx = np.mgrid[0:size, 0:size]
image = np.empty((size, size, 3), dtype=np.uint8)
image[..., 0] = 30 + xx * 25 // (size - 1)
image[..., 1] = 45 + yy * 20 // (size - 1)
image[..., 2] = 65

center = (266, 261)
radii = (112, 92)
shape = ((xx - center[0]) / radii[0]) ** 2 + ((yy - center[1]) / radii[1]) ** 2 <= 1
image[shape] = (205, 115, 85)
highlight = ((xx - 240) / 35) ** 2 + ((yy - 230) / 25) ** 2 <= 1
image[highlight] = (235, 165, 130)

box = [141.2, 156.2, 390.8, 365.8]
points = [[266, 261], [80, 80]]
labels = [1, 0]

In [ ]:
segmenter = SAM2Segmenter(
    checkpoint_path=CHECKPOINT_PATH,
    model_config=MODEL_CONFIG,
    device="cuda",
    dtype="float32",
)
segmenter.load()
segmenter.set_image(image)
box_result = segmenter.segment_with_box(box, multimask_output=True)
points_result = segmenter.segment_with_points(points, labels, multimask_output=True)

In [ ]:
def make_overlay(mask):
    overlay = image.astype(np.float32).copy()
    overlay[mask] = 0.55 * overlay[mask] + 0.45 * np.array([255, 0, 0])
    return np.clip(overlay, 0, 255).astype(np.uint8)

def show_result(result, title):
    overlay = make_overlay(result.selected_mask)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(image)
    axes[0].set_title("Imagem sintética")
    axes[1].imshow(image)
    if result.prompt_type == "box":
        x0, y0, x1, y1 = result.prompt_data["box_xyxy"]
        axes[1].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="lime", linewidth=2))
    else:
        prompt_points = np.asarray(result.prompt_data["points_xy"])
        prompt_labels = np.asarray(result.prompt_data["labels"])
        colors = np.where(prompt_labels == 1, "lime", "red")
        axes[1].scatter(prompt_points[:, 0], prompt_points[:, 1], c=colors, marker="*", s=180, edgecolors="white")
    axes[1].set_title("Prompt")
    axes[2].imshow(result.selected_mask, cmap="gray")
    axes[2].set_title("Máscara selecionada")
    axes[3].imshow(overlay)
    axes[3].set_title(f"Overlay — score {result.selected_score:.4f}")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_result(box_result, "SAM 2.1 — bounding box")
show_result(points_result, "SAM 2.1 — ponto positivo e ponto negativo")

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "outputs" / "sam2_m2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, result in (("box", box_result), ("points", points_result)):
    Image.fromarray(result.selected_mask.astype(np.uint8) * 255, mode="L").save(OUTPUT_DIR / f"{name}_mask.png")
    Image.fromarray(make_overlay(result.selected_mask), mode="RGB").save(OUTPUT_DIR / f"{name}_overlay.png")
    print(
        f"{name}: score={result.selected_score:.6f}, "
        f"load={result.load_time_seconds:.4f}s, "
        f"embedding={result.embedding_time_seconds:.4f}s, "
        f"predict={result.prediction_time_seconds:.4f}s, "
        f"peak_vram={result.peak_vram_bytes} bytes, dtype={result.dtype}"
    )

segmenter.close()